In [1]:
import richdem as rd
import numpy as np
from scipy.ndimage import gaussian_filter
import rasterio

# 1. Load Data using Rasterio to get a numpy array for smoothing
with rasterio.open("dem_UTM.tif") as src:
    dem_data = src.read(1)
    profile = src.profile

# 2. Smooth the DEM
# sigma=1 or 2 is usually enough to kill speckle without losing mountains
# The larger the sigma, the smoother (and flatter) the terrain becomes.
# sigma=1.0 gives the best balance between smoothing and epsilon gradient compatibility
dem_smoothed_np = gaussian_filter(dem_data, sigma=1.0)

# 3. Convert back to RichDEM format
# Handle case where original DEM doesn't have NoData set
nodata_value = profile['nodata'] if profile['nodata'] is not None else -9999
dem_smoothed_rd = rd.rdarray(dem_smoothed_np, no_data=nodata_value)
dem_smoothed_rd.geotransform = [profile['transform'].c, profile['transform'].a, profile['transform'].b, 
                                profile['transform'].f, profile['transform'].d, profile['transform'].e]

# 4. Now Fill Depressions on the SMOOTHED data
dem_filled = rd.FillDepressions(dem_smoothed_rd, epsilon=True, topology='D8')

# 5. Save Result
rd.SaveGDAL("filled_smoothed_topography.tif", dem_filled)

# 6. Copy projection from original DEM (RichDEM strips this)
print("\nCopying projection metadata from original DEM...")
from osgeo import gdal as gdal_copy
src_ds = gdal_copy.Open("dem_UTM.tif")
src_proj = src_ds.GetProjection()
src_ds = None

dst_ds = gdal_copy.Open("filled_smoothed_topography.tif", gdal_copy.GA_Update)
dst_ds.SetProjection(src_proj)
dst_ds = None
print("✓ Projection metadata copied")

# Optional: Check the new sink volume (It should be much less than 1 billion m3)
diff = dem_filled - dem_smoothed_rd
print(f"\nNew Max Sink Depth: {np.max(diff):.2f} m")

/home/sentoki/miniconda3/envs/flower/lib/python3.12/site-packages/richdem/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



Copying projection metadata from original DEM...
✓ Projection metadata copied

New Max Sink Depth: 9.84 m



A Priority-Flood+Epsilon
C Barnes, R., Lehman, C., Mulla, D., 2014. Priority-flood: An optimal depression-filling and watershed-labeling algorithm for digital elevation models. Computers & Geosciences 62, 117–127. doi:10.1016/j.cageo.2013.04.024

c topology = D8
p Setting up boolean flood array matrix...
p Adding cells to the priority queue...
p Performing Priority-Flood+Epsilon...
t succeeded in 2.16423 s======================] (100% - 0.0s - 1 threads)
m Cells processed = 13093200
m Cells in pits = 3828525
W W In assigning negligible gradients to depressions, some depressions rose above the surrounding cells. This implies that a larger storage type should be used. The problem occured for 1975 of 13093200.


# DEM hole counter

In [17]:
import richdem as rd
import numpy as np

def analyze_sinks_richdem(dem_path):
    """
    Analyze sinks (depressions) in a DEM using RichDEM.
    
    Returns statistics about trapped water in depressions.
    """
    # Load data with NoData value specified
    # RichDEM requires explicit NoData value
    try:
        dem = rd.LoadGDAL(dem_path, no_data=-9999)
    except:
        dem = rd.LoadGDAL(dem_path, no_data=-32768)
    
    print(f'DEM dimensions: {dem.shape[0]} x {dem.shape[1]} pixels')
    print(f'Cell size: {abs(dem.geotransform[1]):.2f} x {abs(dem.geotransform[5]):.2f} meters')
    
    # Fill the depressions
    # epsilon=False preserves the flat elevation of the filled sink
    print('\nFilling depressions...')
    dem_filled = rd.FillDepressions(dem, epsilon=False, topology='D8')
    
    # Calculate difference (The depth of trapped water)
    diff = dem_filled - dem
    
    # Convert to numpy for statistical analysis
    diff_np = np.array(diff)
    
    # Identify sinks (anywhere water was added)
    sink_mask = diff_np > 0
    num_sinks = np.sum(sink_mask)
    
    # Calculate volume
    gt = dem.geotransform
    cell_area = abs(gt[1] * gt[5]) 
    total_volume = np.sum(diff_np[sink_mask]) * cell_area

    return {
        "number_of_sinks": num_sinks,
        "percentage_area": (num_sinks / diff_np.size) * 100,
        "max_sink_depth": np.max(diff_np) if num_sinks > 0 else 0,
        "mean_sink_depth": np.mean(diff_np[sink_mask]) if num_sinks > 0 else 0,
        "total_trapped_volume": total_volume,
        "cell_area": cell_area,
        "sink_mask": sink_mask,  # Boolean array for visualization
        "depth_difference": diff_np  # Full difference array
    }

# Run analysis
print('='*70)
print('ANALYZING DEM FOR SINKS USING RICHDEM')
print('='*70)
print()

results = analyze_sinks_richdem("filled_smoothed_topography.tif")

print()
print('='*70)
print('RESULTS')
print('='*70)
print(f'Number of sink pixels: {results["number_of_sinks"]:,}')
print(f'Percentage of DEM area: {results["percentage_area"]:.3f}%')
print(f'Cell area: {results["cell_area"]:.2f} m²')
print()
print(f'Maximum sink depth: {results["max_sink_depth"]:.3f} m')
print(f'Mean sink depth: {results["mean_sink_depth"]:.3f} m')
print()
print(f'Total trapped water volume: {results["total_trapped_volume"]:,.2f} m³')
print(f'                          = {results["total_trapped_volume"]/1e6:.4f} million m³')
print(f'                          = {results["total_trapped_volume"]/1e9:.6f} billion m³')
print('='*70)

ANALYZING DEM FOR SINKS USING RICHDEM

DEM dimensions: 3600 x 3637 pixels
Cell size: 26.52 x 30.90 meters

Filling depressions...

RESULTS
Number of sink pixels: 0
Percentage of DEM area: 0.000%
Cell area: 819.48 m²

Maximum sink depth: 0.000 m
Mean sink depth: 0.000 m

Total trapped water volume: 0.00 m³
                          = 0.0000 million m³
                          = 0.000000 billion m³

RESULTS
Number of sink pixels: 0
Percentage of DEM area: 0.000%
Cell area: 819.48 m²

Maximum sink depth: 0.000 m
Mean sink depth: 0.000 m

Total trapped water volume: 0.00 m³
                          = 0.0000 million m³
                          = 0.000000 billion m³



A Priority-Flood (Zhou2016 version)
C Zhou, G., Sun, Z., Fu, S., 2016. An efficient variant of the Priority-Flood algorithm for filling depressions in raster digital elevation models. Computers & Geosciences 90, Part A, 87 – 96. doi:http://dx.doi.org/10.1016/j.cageo.2016.02.021

t Zhou2016 wall-time = 1.65028 s


In [18]:
# VISUALIZE SINK LOCATIONS

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Check if there are any sinks to visualize
if results['number_of_sinks'] == 0:
    print("\n✓ No sinks found in the DEM - it's completely filled!")
    print("  This is expected for the preprocessed DEM after filling.")
else:
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    
    # Left: Binary sink map (where are the sinks?)
    sink_display = np.ma.masked_where(~results['sink_mask'], results['sink_mask'])
    im1 = ax1.imshow(sink_display, cmap='Reds', interpolation='nearest')
    ax1.set_title('Sink Locations\n(Red = Depression/Sink)', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Pixel X', fontsize=11)
    ax1.set_ylabel('Pixel Y', fontsize=11)
    plt.colorbar(im1, ax=ax1, label='Has Sink')
    
    # Right: Sink depth map (how deep are the sinks?)
    depth_display = np.ma.masked_where(results['depth_difference'] <= 0, results['depth_difference'])
    
    # Set color scale based on actual data
    depths_in_sinks = results['depth_difference'][results['sink_mask']]
    if len(depths_in_sinks) > 0:
        vmax = np.percentile(depths_in_sinks, 99)
    else:
        vmax = results['max_sink_depth']
    
    im2 = ax2.imshow(depth_display, cmap='viridis', interpolation='nearest', 
                    vmin=0, vmax=vmax)
    ax2.set_title('Sink Depth Distribution\n(Brighter = Deeper)', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Pixel X', fontsize=11)
    ax2.set_ylabel('Pixel Y', fontsize=11)
    cbar2 = plt.colorbar(im2, ax=ax2, label='Depth (m)')
    
    plt.tight_layout()
    plt.savefig('outputs/dem_sink_analysis.png', dpi=150, bbox_inches='tight')
    print("\n✓ Visualization saved: outputs/dem_sink_analysis.png")
    plt.show()
    
    # Print some additional statistics
    print("\n" + "="*70)
    print("SINK DEPTH DISTRIBUTION")
    print("="*70)
    depths = results['depth_difference'][results['sink_mask']]
    if len(depths) > 0:
        print(f"Min sink depth: {depths.min():.3f} m")
        print(f"25th percentile: {np.percentile(depths, 25):.3f} m")
        print(f"50th percentile (median): {np.percentile(depths, 50):.3f} m")
        print(f"75th percentile: {np.percentile(depths, 75):.3f} m")
        print(f"95th percentile: {np.percentile(depths, 95):.3f} m")
        print(f"99th percentile: {np.percentile(depths, 99):.3f} m")
        print(f"Max sink depth: {depths.max():.3f} m")
    else:
        print("No sinks found!")
    print("="*70)


✓ No sinks found in the DEM - it's completely filled!
  This is expected for the preprocessed DEM after filling.


## 📊 Sink Analysis Summary

Your DEM has **significant sink/depression issues**:

### Key Findings:
- **34.2%** of your DEM consists of sinks (depressions that trap water)
- **4.48 million pixels** are affected
- Maximum sink depth: **16.77 meters** 
- Total trapped volume: **1.06 billion m³** (over 1 cubic kilometer!)

### What This Means:
In a real hydrological simulation, water flowing into these sinks would get trapped and not drain naturally. This can cause:
1. **Unrealistic ponding** - Water accumulates indefinitely
2. **Flow interruption** - Natural drainage patterns are blocked
3. **Simulation instability** - ANUGA may struggle with flat areas

### Recommendations:
1. **Fill the depressions** before using in ANUGA (see first cell)
2. **Use epsilon=True** when filling to add tiny gradients for flow
3. **Verify results** - Check if filled DEM looks reasonable for your study area

The deep sinks (>16m) might indicate:
- Data artifacts/errors in the original DEM
- Bridge/culvert locations that appear as depressions
- Actual geological features (sinkholes, quarries)